# Cloud-Based Intelligent Resource Allocation System for Campus Facilities (Phase 1)

**Domain:** Education

**Framework:** AutoGen (pyautogen)

**Model:** GPT-4o-mini

**Approach:**  Predefined Sequence 

**Purpose:** Waterfall SDLC simulation using AutoGen Predefined Sequence

## Environment Setup

Install required libraries, configure the OpenAI API key, and initialize the AutoGen LLM settings.

In [1]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "pyautogen<0.4", "openai", "tiktoken", "--quiet"
])
print("Libraries installed.")

Libraries installed.


In [2]:
import os, json, time
from datetime import datetime
import autogen
from autogen import AssistantAgent, UserProxyAgent
import tiktoken

# Load API key from .env file
env_path = os.path.join(os.getcwd(), ".env")
if os.path.exists(env_path):
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found. Create a .env file with: OPENAI_API_KEY=sk-...")

MODEL = "gpt-4o-mini"

config_list = [{
    "model": MODEL,
    "api_key": OPENAI_API_KEY,
}]

llm_config = {
    "config_list": config_list,
    "temperature": 0.3,
    "max_tokens": 8192,
}

# Tiktoken for accurate token measurement
enc = tiktoken.encoding_for_model(MODEL)

print(f"API Key loaded. Model: {MODEL}")
print(f"Framework: AutoGen (pyautogen v{autogen.__version__})")

API Key loaded. Model: gpt-4o-mini
Framework: AutoGen (pyautogen v0.3.2)


## Project Configuration

Project details and realistic system boundaries are defined here. Appropriate scope is defined and shared with the engineering agents.

In [3]:
DOMAIN = """
PROJECT: Cloud-Based Intelligent Resource Allocation System for Campus Facilities
DOMAIN: Education
DESCRIPTION: Students and faculty book rooms, labs, and equipment online.
Administrators view real-time occupancy dashboards. AI/ML predicts demand and
auto-allocates resources. Integrates with Google Calendar, Microsoft Outlook,
and SSO (Single Sign-On). Sends booking/conflict notifications. Generates
utilization reports for facility management. Supports 5,000+ concurrent users
and maintains 99.9% uptime.
STAKEHOLDERS: Students, Faculty, Facility Admins, IT Department, University Management
FACILITIES: Classrooms, Computer Labs, Research Labs, Conference Rooms, Study Rooms
"""

PROJECT_SCOPE = """
Project scope and implementation boundaries:
- Total SLOC must be between 12,000 and 22,000
- This is a mid-size cloud-based campus facility management system
- Frontend (React/Next.js): 3,000–5,000 SLOC
- Backend API (Node.js/FastAPI): 4,000–7,000 SLOC
- AI/ML Module: 2,000–4,000 SLOC
- Database layer: 1,500–2,500 SLOC
- Integration layer (SSO, Calendar): 1,000–2,000 SLOC

CONSISTENCY CHECK before printing metrics:
  SLOC-based days  = Total SLOC ÷ 50
  Hours-based days = Sum of all task hours ÷ 8
  These two numbers must be within 15% of each other.
  If not, fix task hours first. Then print:
  CONSISTENCY CHECK: [SLOC days] ≈ [Hours days] ✓
"""

print("Project configuration loaded.")

Project configuration loaded.


## Define All AI Agents
Define all 7 AI agents for the Waterfall SDLC simulation.

- **Customer Proxy** (UserProxyAgent) — sends the problem statement to the Project Manager
- **Project Manager** (AssistantAgent) — receives and sends messages to each engineering agent
- **Requirements Engineer** (AssistantAgent) — generates use cases, functional and non-functional requirements
- **System Engineer** (AssistantAgent) — creates system architecture and design document
- **Software Developer** (AssistantAgent) — defines development tasks and SLOC estimates
- **Test Engineer** (AssistantAgent) — creates test cases and testing strategy
- **Documentation Engineer** (AssistantAgent) — creates end-user and technical documentation plan

In [4]:
# ── Agent 1: Customer Proxy (UserProxyAgent – human-in-the-loop) ──────────────
customer_proxy = UserProxyAgent(
    name="Customer_Proxy",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=0,
    code_execution_config=False,
    default_auto_reply="",
)

# ── Agent 2: Project Manager (AssistantAgent) ─────────────────────────────────
project_manager = AssistantAgent(
    name="Project_Manager",
    system_message=f"""You are the Project Manager AI Agent for the Waterfall SDLC.
You have received the customer request. Briefly acknowledge it, state the 5 Waterfall
phases that will be executed (Requirements, Design, Implementation, Testing, Documentation),
and confirm you are delegating to each specialist agent in order.
Keep your response concise – under 200 words.
{DOMAIN}""",
    llm_config=llm_config,
)

# ── Agent 3: Requirements Engineer ───────────────────────────────────────────
requirement_engineer = AssistantAgent(
    name="Requirement_Engineer",
    system_message=f"""You are the Requirements Engineer AI Agent for the Waterfall SDLC.
{DOMAIN}

Your responsibilities:
1. Create a COMPLETE use case list with unique IDs (UC-001, UC-002, ...) – at least 15 use cases
2. For each use case provide: Use Case ID, Name, Primary Actor, Preconditions,
   Postconditions, Main Flow, Alternate Flows
3. Create FUNCTIONAL requirements (FR-001, FR-002, ...) – at least 40 requirements
4. Create NON-FUNCTIONAL requirements (NFR-001, NFR-002, ...) – at least 15 requirements
5. All requirement IDs are sequential across the entire document
6. Include review and rework tasks (10% buffer)

Then calculate effort EXACTLY as follows:
- Step 1: Work = Total FR + NFR only (do NOT include use cases in the count)
- Step 2: Productivity rate = 5 requirements per day
- Step 3: Effort = Work / Productivity rate
- Step 4: Review effort = Effort × 10%
- Step 5: Total Effort WITH Review = Effort + Review effort
- Step 6: Earliest Start = Day 1
- Step 7: Earliest Finish = Day 1 + Total Effort WITH Review - 1 (round up to whole number)

End your response with EXACTLY this block (real numbers only, no placeholders):
=== REQUIREMENTS PHASE METRICS ===
Total Use Cases: 
Total Functional Requirements (FR): 
Total Non-Functional Requirements (NFR): 
Total Requirements (FR + NFR): 
Productivity Rate: 5 requirements per day
Base Effort: [Total REQs] / 5 = [X] days
Review Effort (10%): [X] days
Total Effort WITH Review: [X] days
Earliest Start: Day 1
Earliest Finish: Day [X]
Number of Tasks in Requirements Phase: 
================================""",
    llm_config=llm_config,
)

# ── Agent 4: System Engineer ──────────────────────────────────────────────────
system_engineer = AssistantAgent(
    name="System_Engineer",
    system_message=f"""You are the System Engineer AI Agent for the Waterfall SDLC.
{DOMAIN}

Your responsibilities:
1. Create a HIGH-LEVEL SYSTEM ARCHITECTURE with layered design:
   (Client Layer, API/Orchestration Layer, AI/Intelligence Layer,
    Storage Layer, Cross-Cutting Services)
2. Provide a COMPONENT DIAGRAM (textual description)
3. Provide a SEQUENCE DIAGRAM for the Room Booking use case
4. Create an ENTITY-RELATIONSHIP DIAGRAM (textual) with all entities
5. Describe the TECHNOLOGY STACK recommendations
6. Design the AI/ML demand prediction module architecture
7. Design SSO and Calendar integration architecture
8. Reference specific FR and NFR IDs from the requirements document
9. Include review and rework tasks (10% buffer)

Then calculate effort EXACTLY:
- Step 1: Work = estimated total number of pages in design document
- Step 2: Productivity rate = 5 pages per day
- Step 3: Effort = Work / Productivity rate
- Step 4: Review effort = Effort × 10%
- Step 5: Total Effort WITH Review = Effort + Review effort
- Step 6: Earliest Start = Day after Requirements Earliest Finish
- Step 7: Earliest Finish = Earliest Start + Total Effort WITH Review - 1 (round up to whole number)

End with EXACTLY:
=== DESIGN PHASE METRICS ===
Total Design Document Pages: 
Architecture Diagrams Count: 
Database Schema Pages: 
API Documentation Pages: 
UI Wireframe Pages: 
Other Design Pages: 
Productivity Rate: 5 pages per day
Base Effort: [Total Pages] / 5 = [X] days
Review Effort (10%): [X] days
Total Effort WITH Review: [X] days
Earliest Start: Day [X]
Earliest Finish: Day [X]
Number of Tasks in Design Phase: 
=============================""",
    llm_config=llm_config,
)

# ── Agent 5: Software Developer ───────────────────────────────────────────────
software_developer = AssistantAgent(
    name="Software_Developer",
    system_message=f"""You are the Software Developer AI Agent for the Waterfall SDLC.
{DOMAIN}
{PROJECT_SCOPE}

Your responsibilities:
1. List ALL development tasks with IDs (T-001, T-002, ...)
2. For each task: ID, Module, Description, SLOC estimate, Hours estimate, Linked FR IDs
3. Organize tasks by module:
   - User Auth & SSO Module
   - Facility Booking Module
   - Real-time Occupancy Dashboard
   - AI Demand Prediction Module
   - Notification & Alerts Module
   - Reporting & Analytics Module
   - Admin Management Module
   - API Gateway & Integration
4. Minimum 25 tasks, maximum 45 tasks
5. Include code review tasks (10% buffer)

Then calculate effort EXACTLY:
- Step 1: Work = total SLOC (MUST be between 12,000 and 22,000)
- Step 2: Productivity rate = 50 SLOC per day
- Step 3: Effort = Work / Productivity rate
- Step 4: Review effort = Effort × 10%
- Step 5: Total Effort WITH Review = Effort + Review effort
- Step 6: Sum all task hours once by adding each task's hours estimate. Do NOT repeat any numbers. Print only the final total sum. Then check: SLOC/50 × 8 ≈ Total Task Hours
- Step 7: Earliest Start = Day after Design Earliest Finish
- Step 8: Earliest Finish = Earliest Start + Total Effort WITH Review - 1 (round up to whole number)

End with EXACTLY:
=== IMPLEMENTATION PHASE METRICS ===
Total Development Tasks: 
Frontend SLOC: 
Backend SLOC: 
Database Scripts SLOC: 
AI/ML Module SLOC: 
Integration/API SLOC: 
Total SLOC (MUST be 12,000–22,000): 
Productivity Rate: 50 SLOC per day
Base Effort: [Total SLOC] / 50 = [X] days
Code Review Effort (10%): [X] days
Total Effort WITH Review: [X] days
Total Task Hours (sum of all tasks): 
CONSISTENCY CHECK: [SLOC/50 × 8] = [X] hrs ≈ Total Task Hours [PASS/FAIL]
Earliest Start: Day [X]
Earliest Finish: Day [X]
Number of Tasks in Implementation Phase: 
===================================""",
    llm_config=llm_config,
)

# ── Agent 6: Test Engineer ────────────────────────────────────────────────────
test_engineer = AssistantAgent(
    name="Test_Engineer",
    system_message=f"""You are the Test Engineer AI Agent for the Waterfall SDLC.
{DOMAIN}

Your responsibilities:
1. Define the overall TEST STRATEGY (levels: unit, integration, system, AI evaluation)
2. Create FUNCTIONAL TEST CASES (TC-001 through TC-XX) with:
   Related UC/FR IDs, Preconditions, Steps, Expected Results
3. Create NON-FUNCTIONAL TEST CASES (NTC-001 through NTC-XX) covering:
   Performance (5,000 concurrent users), AI Accuracy (>85% target),
   Availability (99.9% uptime), Security/RBAC, Usability
4. Provide a TESTING SCHEDULE broken into weeks
5. Include test review tasks (10% buffer)

Then calculate effort EXACTLY:
- Step 1: Work = total number of test cases
- Step 2: Productivity rate = 2 test cases per day
- Step 3: Effort = Work / Productivity rate
- Step 4: Review effort = Effort × 10%
- Step 5: Total Effort WITH Review = Effort + Review effort
- Step 6: Earliest Start = Day after Implementation Earliest Finish
- Step 7: Earliest Finish = Earliest Start + Total Effort WITH Review - 1 (round up to whole number)

End with EXACTLY:
=== TESTING PHASE METRICS ===
Total Test Cases: 
Functional Test Cases (TC): 
Non-Functional Test Cases (NTC): 
Unit Test Cases: 
Integration Test Cases: 
System/E2E Test Cases: 
Performance Test Cases: 
AI Accuracy Test Cases: 
UAT Test Cases: 
Productivity Rate: 2 test cases per day
Base Effort: [Total TCs] / 2 = [X] days
Review Effort (10%): [X] days
Total Effort WITH Review: [X] days
Earliest Start: Day [X]
Earliest Finish: Day [X]
Number of Tasks in Testing Phase: 
=============================""",
    llm_config=llm_config,
)

# ── Agent 7: Documentation Engineer ──────────────────────────────────────────
documentation_engineer = AssistantAgent(
    name="Documentation_Engineer",
    system_message=f"""You are the Documentation Engineer AI Agent for the Waterfall SDLC.
{DOMAIN}

Your responsibilities:
1. Create an END-USER DOCUMENTATION OUTLINE:
   - Student User Guide (booking, equipment, notifications)
   - Faculty User Guide (booking, recurring reservations)
   - Administrator Guide (dashboard, reports, AI settings)
   - AI System Explainer (how demand prediction works)
   - Troubleshooting & FAQ
2. Create a DEVELOPER DOCUMENTATION OUTLINE:
   - System Architecture Overview
   - RESTful & GraphQL API Reference
   - Database Schema & ERD
   - AI/ML Model Documentation
   - Deployment & CI/CD Guide
   - Security & Compliance Guide
3. Create TRAINING MATERIALS outline:
   - Student Onboarding (pages estimate)
   - Admin Training Manual (pages estimate)
4. Estimate pages per section
5. Include documentation review tasks (10% buffer)

Then calculate effort EXACTLY:
- Step 1: Work = total documentation pages
- Step 2: Productivity rate = 3 pages per day
- Step 3: Effort = Work / Productivity rate
- Step 4: Review effort = Effort × 10%
- Step 5: Total Effort WITH Review = Effort + Review effort
- Step 6: Earliest Start = Day after Testing Earliest Finish
- Step 7: Earliest Finish = Earliest Start + Total Effort WITH Review - 1 (round up to whole number)

End with EXACTLY:
=== DOCUMENTATION PHASE METRICS ===
End-User Documentation Pages: 
Technical/Developer Documentation Pages: 
Training Material Pages: 
Total Documentation Pages: 
Productivity Rate: 3 pages per day
Base Effort: [Total Pages] / 3 = [X] days
Review Effort (10%): [X] days
Total Effort WITH Review: [X] days
Earliest Start: Day [X]
Earliest Finish: Day [X]
Number of Tasks in Documentation Phase: 
===================================""",
    llm_config=llm_config,
)

print("All 7 agents created successfully.")
print("  - Customer_Proxy       (UserProxyAgent)")
print("  - Project_Manager      (AssistantAgent)")
print("  - Requirement_Engineer (AssistantAgent)")
print("  - System_Engineer      (AssistantAgent)")
print("  - Software_Developer   (AssistantAgent)")
print("  - Test_Engineer        (AssistantAgent)")
print("  - Documentation_Engineer (AssistantAgent)")

All 7 agents created successfully.
  - Customer_Proxy       (UserProxyAgent)
  - Project_Manager      (AssistantAgent)
  - Requirement_Engineer (AssistantAgent)
  - System_Engineer      (AssistantAgent)
  - Software_Developer   (AssistantAgent)
  - Test_Engineer        (AssistantAgent)
  - Documentation_Engineer (AssistantAgent)


## Customer Problem Statement

The customer sends the initial requirements to the Project Manager.

In [5]:
customer_message = (
    "I want to create a Cloud-Based Intelligent Resource Allocation System "
    "for Campus Facilities at our university. The system must:\n"
    "1. Allow students and faculty to book rooms, labs, and equipment online\n"
    "2. Allow administrators to view real-time occupancy dashboards\n"
    "3. Use AI/ML to predict demand and auto-allocate resources\n"
    "4. Integrate with Google Calendar, Microsoft Outlook, and SSO\n"
    "5. Send notifications for booking confirmations and conflicts\n"
    "6. Generate utilization reports for facility management\n"
    "7. Support 5,000+ concurrent users and maintain 99.9% uptime\n"
    "Please generate the complete Waterfall project plan."
)

print("Customer message defined.")
print(f"Message length: {len(customer_message)} characters")

Customer message defined.
Message length: 615 characters


## Run Experiment

The Project Manager initiates the conversation with each engineering agent in sequence, passing the relevant context from previous agents to generate the complete Waterfall project plan.

In [6]:
import time, os, json
from datetime import datetime

print("Starting AutoGen Predefined Sequence experiment...")
print("=" * 60)

start_time = time.time()

customer_message = (
    "I want to create a Cloud-Based Intelligent Resource Allocation System "
    "for Campus Facilities at our university. The system must:\n"
    "1. Allow students and faculty to book rooms, labs, and equipment online\n"
    "2. Allow administrators to view real-time occupancy dashboards\n"
    "3. Use AI/ML to predict demand and auto-allocate resources\n"
    "4. Integrate with Google Calendar, Microsoft Outlook, and SSO\n"
    "5. Send notifications for booking confirmations and conflicts\n"
    "6. Generate utilization reports for facility management\n"
    "7. Support 5,000+ concurrent users and maintain 99.9% uptime\n"
    "Please generate the complete Waterfall project plan."
)

all_responses = {}

# ── Step 1: Customer Proxy → Project Manager ──────────────────────────────
print("\nStep 1: Customer Proxy → Project Manager")
pm_chat = customer_proxy.initiate_chat(
    recipient=project_manager,
    message=customer_message,
    max_turns=1,
    summary_method="last_msg"
)
pm_response = pm_chat.summary
all_responses["Project_Manager"] = pm_response
print(f"Project Manager responded ({len(pm_response)} chars)")

# ── Step 2: Project Manager → Requirements Engineer ───────────────────────
print("\nStep 2: Project Manager → Requirements Engineer")
pm_to_re = (
    f"I have received the following message from the customer:\n"
    f'"{customer_message}"\n\n'
    f"Based on this message, generate a complete list of detailed use cases "
    f"and requirements. Give every use case a unique name and number (UC-001, UC-002...) "
    f"and give every requirement a unique sequential number (REQ-001, REQ-002...) "
    f"across all use cases. Include at least 15 use cases, 40 functional requirements, "
    f"and 15 non-functional requirements. "
    f"Provide estimates about the productivity of the requirements engineer. "
    f"What are the different tasks needed for the requirements phase? "
    f"Include reviews and rework tasks. "
    f"Calculate effort using: Work / Productivity Rate. "
    f"Print all metrics at the end."
)
re_chat = project_manager.initiate_chat(
    recipient=requirement_engineer,
    message=pm_to_re,
    max_turns=1,
    summary_method="last_msg"
)
re_response = re_chat.summary
all_responses["Requirement_Engineer"] = re_response
print(f"Requirements Engineer responded ({len(re_response)} chars)")

# ── Step 3: Project Manager → System Engineer ─────────────────────────────
print("\nStep 3: Project Manager → System Engineer")
pm_to_se = (
    f"I have received the following message from the customer:\n"
    f'"{customer_message}"\n\n'
    f"I sent the message to the Requirements Engineer who gave me the detailed "
    f"requirements document. Here is the requirements document including metrics:\n"
    f"{re_response}\n\n"
    f"Based on the customer request and the requirements engineer input, "
    f"create a complete system design document. Include architecture, component diagram, "
    f"sequence diagram, ERD, technology stack, AI/ML module design, and SSO integration. "
    f"Provide estimates about the productivity of the system engineer. "
    f"How many pages and diagrams can the engineer complete every day? "
    f"What are the different tasks needed for the design phase? "
    f"Include reviews and rework tasks. "
    f"Calculate effort using: Work / Productivity Rate. "
    f"Print all metrics at the end. "
    f"IMPORTANT: The Requirements phase Earliest Finish is stated in the metrics above. "
    f"Your Design phase Earliest Start must be the day AFTER that date."
)
se_chat = project_manager.initiate_chat(
    recipient=system_engineer,
    message=pm_to_se,
    max_turns=1,
    summary_method="last_msg"
)
se_response = se_chat.summary
all_responses["System_Engineer"] = se_response
print(f"System Engineer responded ({len(se_response)} chars)")

# ── Step 4: Project Manager → Software Developer ──────────────────────────
print("\nStep 4: Project Manager → Software Developer")
pm_to_dev = (
    f"The Requirements Engineer and System Engineer have completed their work. "
    f"Requirements document with metrics:\n{re_response}\n\n"
    f"Design document with metrics:\n{se_response}\n\n"
    f"Based on the requirements and design documents, create a complete "
    f"implementation plan with all development tasks. "
    f"List every task with a unique ID (T-001, T-002...), module, description, "
    f"SLOC estimate, hours estimate, and linked requirement IDs. "
    f"Provide estimates about the productivity of the software developer. "
    f"How many lines of code can the developer write every day? "
    f"What are the different tasks needed for the implementation phase? "
    f"Include code review tasks. "
    f"Calculate effort using: Work / Productivity Rate. "
    f"Print all metrics at the end."
    f"IMPORTANT: The Design phase Earliest Finish is stated in the metrics above. "
    f"Your Implementation phase Earliest Start must be the day AFTER that date."
)
dev_chat = project_manager.initiate_chat(
    recipient=software_developer,
    message=pm_to_dev,
    max_turns=1,
    summary_method="last_msg"
)
dev_response = dev_chat.summary
all_responses["Software_Developer"] = dev_response
print(f"Software Developer responded ({len(dev_response)} chars)")

# ── Step 5: Project Manager → Test Engineer ───────────────────────────────
print("\nStep 5: Project Manager → Test Engineer")
pm_to_te = (
    f"The software development plan is complete. "
    f"Requirements document with metrics:\n{re_response}\n\n"
    f"Implementation document with metrics:\n{dev_response}\n\n"
    f"Based on the requirements and implementation documents, create a complete "
    f"test plan with all test cases (TC-001, TC-002...). "
    f"Include unit, integration, system, performance, AI accuracy (target >85%), "
    f"and UAT test cases. "
    f"Provide estimates about the productivity of the test engineer. "
    f"How many test cases can the engineer create and execute every day? "
    f"What are the different tasks needed for the testing phase? "
    f"Include test review tasks. "
    f"Calculate effort using: Work / Productivity Rate. "
    f"Print all metrics at the end."
    f"IMPORTANT: The Implementation phase Earliest Finish is stated in the metrics above. "
    f"Your Testing phase Earliest Start must be the day AFTER that date."
)
te_chat = project_manager.initiate_chat(
    recipient=test_engineer,
    message=pm_to_te,
    max_turns=1,
    summary_method="last_msg"
)
te_response = te_chat.summary
all_responses["Test_Engineer"] = te_response
print(f"Test Engineer responded ({len(te_response)} chars)")

# ── Step 6: Project Manager → Documentation Engineer ─────────────────────
print("\nStep 6: Project Manager → Documentation Engineer")
pm_to_doc = (
    f"Testing is complete. Here are the metrics from all previous phases:\n"
    f"Requirements metrics:\n{re_response[-500:]}\n\n"
    f"Design metrics:\n{se_response[-500:]}\n\n"
    f"Implementation metrics:\n{dev_response[-500:]}\n\n"
    f"Testing metrics:\n{te_response[-500:]}\n\n"
    f"Based on all previous documents, create a complete documentation plan. "
    f"Include end-user documentation (Student Guide, Faculty Guide, Admin Guide, "
    f"AI System Explainer, FAQ), developer documentation (Architecture, API Reference, "
    f"DB Schema, AI/ML Docs, CI/CD, Security), and training materials. "
    f"Estimate pages per section. "
    f"Provide estimates about the productivity of the documentation engineer. "
    f"How many pages can the engineer complete every day? "
    f"What are the different tasks needed for the documentation phase? "
    f"Include documentation review tasks. "
    f"Calculate effort using: Work / Productivity Rate. "
    f"Print all metrics at the end."
    f"IMPORTANT: The Testing phase Earliest Finish is stated in the metrics above. "
    f"Your Documentation phase Earliest Start must be the day AFTER that date."
)
doc_chat = project_manager.initiate_chat(
    recipient=documentation_engineer,
    message=pm_to_doc,
    max_turns=1,
    summary_method="last_msg"
)
doc_response = doc_chat.summary
all_responses["Documentation_Engineer"] = doc_response
print(f"Documentation Engineer responded ({len(doc_response)} chars)")

total_elapsed = time.time() - start_time
print(f"\n{'=' * 60}")
print(f"Predefined sequence completed in {total_elapsed:.1f}s")
print(f"{'=' * 60}")

Starting AutoGen Predefined Sequence experiment...

Step 1: Customer Proxy → Project Manager
Customer_Proxy (to Project_Manager):

I want to create a Cloud-Based Intelligent Resource Allocation System for Campus Facilities at our university. The system must:
1. Allow students and faculty to book rooms, labs, and equipment online
2. Allow administrators to view real-time occupancy dashboards
3. Use AI/ML to predict demand and auto-allocate resources
4. Integrate with Google Calendar, Microsoft Outlook, and SSO
5. Send notifications for booking confirmations and conflicts
6. Generate utilization reports for facility management
7. Support 5,000+ concurrent users and maintain 99.9% uptime
Please generate the complete Waterfall project plan.

--------------------------------------------------------------------------------
Project_Manager (to Customer_Proxy):

Thank you for your request regarding the Cloud-Based Intelligent Resource Allocation System for Campus Facilities. We will execute th

## Save Results 

Save each agent's output as a markdown file and count tokens per agent using tiktoken.

In [7]:
import os, re

output_dir = "Phase1_Experiment1_Results_AutoGen"
os.makedirs(output_dir, exist_ok=True)

agent_names = [
    "Project_Manager",
    "Requirement_Engineer",
    "System_Engineer",
    "Software_Developer",
    "Test_Engineer",
    "Documentation_Engineer",
]

# System prompts for token counting
agent_system_prompts = {
    a.name: a.system_message
    for a in [project_manager, requirement_engineer, system_engineer,
              software_developer, test_engineer, documentation_engineer]
    if hasattr(a, "system_message") and a.system_message
}

all_results = []
for agent_name, response in all_responses.items():
    output_tokens = len(enc.encode(response))
    sys_prompt = agent_system_prompts.get(agent_name, "")
    input_tokens = len(enc.encode(sys_prompt)) + len(enc.encode(customer_message))
    total_tokens_agent = input_tokens + output_tokens
    all_results.append({
        "agent": agent_name,
        "response": response,
        "tokens": total_tokens_agent,
        "model": MODEL,
    })

total_tokens = sum(r["tokens"] for r in all_results)
        
# Save individual agent outputs as .md files
for result in all_results:
    filepath = os.path.join(output_dir, f"{result['agent']}_output.md")
    with open(filepath, "w") as f:
        f.write(f"# {result['agent'].replace('_', ' ')} Output\n\n")
        f.write(f"**Model:** {result['model']} | **Tokens:** {result['tokens']:,}\n\n")
        f.write(result["response"])
    print(f"Saved: {filepath}")

# Save combined report
section_names = {
    "Project_Manager":        "SECTION 1 – PROJECT MANAGER",
    "Requirement_Engineer":   "SECTION 2 – REQUIREMENTS ENGINEER",
    "System_Engineer":        "SECTION 3 – SYSTEM ENGINEER",
    "Software_Developer":     "SECTION 4 – SOFTWARE DEVELOPER",
    "Test_Engineer":          "SECTION 5 – TEST ENGINEER",
    "Documentation_Engineer": "SECTION 6 – DOCUMENTATION ENGINEER",
}
combined_path = os.path.join(output_dir, "Phase1_Waterfall_Complete_Report.md")
with open(combined_path, "w") as f:
    f.write("# Phase 1 – Waterfall Project Plan\n")
    f.write("# Cloud-Based Intelligent Resource Allocation System for Campus Facilities\n\n")
    f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"**LLM Model:** {MODEL}\n")
    f.write(f"**Framework:** AutoGen (pyautogen) – Predefined Sequence\n\n---\n\n")
    f.write("## SECTION 0 – CUSTOMER PROBLEM STATEMENT\n\n")
    f.write(customer_message.strip() + "\n\n---\n\n")
    for result in all_results:
        agent = result["agent"]
        if agent in section_names:
            f.write(f"## {section_names[agent]}\n\n")
            f.write(f"**Model:** {result['model']} | **Tokens:** {result['tokens']:,}\n\n")
            f.write(result["response"])
            f.write("\n\n---\n\n")

print(f"\nSaved combined report: {combined_path}")

# Save metadata JSON
metadata = {
    "experiment": {
        "title": "Phase 1 – Waterfall Project Planning",
        "project": "Cloud-Based Intelligent Resource Allocation System for Campus Facilities",
        "model": MODEL,
        "framework": f"AutoGen (pyautogen v{autogen.__version__}) – Predefined Sequence",
        "interaction_type": "Predefined Sequence (PM delegates explicitly)",
        "temperature": 0.3,
        "date": datetime.now().isoformat(),
        "total_tokens": total_tokens,
        "total_time_seconds": round(total_elapsed, 2),
    },
    "agents": [{"name": r["agent"], "tokens": r["tokens"], "model": r["model"]} for r in all_results],
    "sequence": {
    "type": "predefined",
    "total_agents": len(all_results),
    "interaction": "PM explicitly delegates to each agent",
    },
}
metadata_path = os.path.join(output_dir, "experiment_metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"Saved metadata: {metadata_path}")

Saved: Phase1_Experiment1_Results_AutoGen/Project_Manager_output.md
Saved: Phase1_Experiment1_Results_AutoGen/Requirement_Engineer_output.md
Saved: Phase1_Experiment1_Results_AutoGen/System_Engineer_output.md
Saved: Phase1_Experiment1_Results_AutoGen/Software_Developer_output.md
Saved: Phase1_Experiment1_Results_AutoGen/Test_Engineer_output.md
Saved: Phase1_Experiment1_Results_AutoGen/Documentation_Engineer_output.md

Saved combined report: Phase1_Experiment1_Results_AutoGen/Phase1_Waterfall_Complete_Report.md
Saved metadata: Phase1_Experiment1_Results_AutoGen/experiment_metadata.json


## Experiment Summary

Gives the final experiment results and per-agent token breakdown.

In [8]:
print("=" * 60)
print("  EXPERIMENT 1 COMPLETE")
print("=" * 60)
print(f"  Model:       {MODEL}")
print(f"  Framework:   AutoGen (pyautogen v{autogen.__version__})")
print(f"  Sequence:    Predefined Sequence")
print(f"  Total Agents: {len(all_results)}")
print(f"  Total Steps:  6 (one per Waterfall phase)")
print(f"  Total Tokens: {total_tokens:,}")
print(f"  Total Time:  {total_elapsed:.1f}s")
print()
print("  Agent Breakdown:")
for r in all_results:
    print(f"    {r['agent']:30s} | Tokens: {r['tokens']:6,} | Response: {len(r['response']):6,} chars")
print()
print(f"  Output Directory: {output_dir}/")
print("=" * 60)

  EXPERIMENT 1 COMPLETE
  Model:       gpt-4o-mini
  Framework:   AutoGen (pyautogen v0.3.2)
  Sequence:    Predefined Sequence
  Total Agents: 6
  Total Steps:  6 (one per Waterfall phase)
  Total Tokens: 13,504
  Total Time:  222.9s

  Agent Breakdown:
    Project_Manager                | Tokens:    510 | Response:    941 chars
    Requirement_Engineer           | Tokens:  4,001 | Response: 14,459 chars
    System_Engineer                | Tokens:  2,300 | Response:  6,594 chars
    Software_Developer             | Tokens:  3,169 | Response:  7,206 chars
    Test_Engineer                  | Tokens:  2,174 | Response:  5,693 chars
    Documentation_Engineer         | Tokens:  1,350 | Response:  2,447 chars

  Output Directory: Phase1_Experiment1_Results_AutoGen/
